# NB5c — Frozen probes for AraELECTRA and CAMeLBERT-MSA

Two more neural rungs of the comparison ladder, using the same protocol as NB5b for AraBERT: freeze
the encoder, extract the [CLS] vector for every article with the same sentence-aware chunk-and-pool,
and score a linear probe against the pair-aware test split. **Nothing about the classification head,
the chunking, the pooling, or the evaluation split changes from NB5b** — that is deliberate. Any
difference between the three probe numbers must come from the encoder alone; if I varied preprocessing
or pooling between them, the comparison would not tell me which encoder is better, only which pipeline
choice is better.

**Input:** `dataset.parquet` (7,101 articles, pair-aware split).
**Output per model:** `nb5c_<name>_cls_frozen.npy` (cached [CLS] vectors), plus a combined
`nb5c_results.parquet` with the standard five metrics for each.

## Two encoders in this notebook

- **AraELECTRA** (`aubmindlab/araelectra-base-discriminator`) — shares the pre-training corpus with
  AraBERTv2 (77 GB, same OSCAR filtering), but uses discriminative rather than masked-LM
  pre-training. Same 768-dimensional [CLS] as the other two.
- **CAMeLBERT-MSA** (`CAMeL-Lab/bert-base-arabic-camelbert-msa`) — pre-trained specifically on MSA
  news, which is exactly the sub-genre of our corpus. Same 768-dimensional [CLS].

## Preprocessing note (worth stating explicitly)

Both encoders were pre-trained **without Farasa segmentation**, unlike AraBERTv2. So the pipeline
here is cleaner and considerably faster (Farasa segmentation cost about 200 seconds in NB5b, and I
skip it here). Concretely: I use `ArabertPreprocessor` in its non-segmenting mode for AraELECTRA, and
a lightweight punctuation-and-diacritic clean for CAMeLBERT. Chunking and pooling are byte-identical
to NB5b.

## Setup

In [1]:
!pip -q install transformers arabert >/dev/null 2>&1

import pandas as pd, numpy as np, re, os, glob, time, json
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'cpu')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
OUT_DIR = '/kaggle/working'

MAX_LEN     = 512
CHUNK_TOK   = 510            # reserve room for [CLS] and [SEP]
OVERLAP_SENTS = 1
MAX_CHUNKS  = 6

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred): return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA_PATH = find_parquet('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', 'dataset')
df = pd.read_parquet(DATA_PATH)
print('dataset:', df.shape, '| splits:', df['split'].value_counts().to_dict())

device: cuda | Tesla T4
dataset: (7101, 7) | splits: {'train': 5363, 'test': 1093, 'val': 645}


## Shared chunking and pooling — byte-identical to NB5b

Same sentence splitter, same overlap policy, same 6-chunk cap, same mean-pool. The only per-model
piece is the tokenizer, which I pass in as an argument so both encoders can reuse this code.

In [2]:
_SENT_SPLIT = re.compile(r'(?<=[.!?\u061f])\s+')

def split_sentences(text):
    return [s.strip() for s in _SENT_SPLIT.split(text.strip()) if s.strip()]

def chunk_token_windows(text, tokenizer):
    sents = split_sentences(text)
    if not sents:
        sents = [text.strip() or tokenizer.unk_token]
    sent_ids = [tokenizer.encode(s, add_special_tokens=False) for s in sents]

    windows, cur, cur_len, i = [], [], 0, 0
    cur_sents = []
    while i < len(sent_ids):
        sid = sent_ids[i]
        if len(sid) > CHUNK_TOK:
            if cur:
                windows.append(cur); cur, cur_len, cur_sents = [], 0, []
                if len(windows) >= MAX_CHUNKS: break
            for s in range(0, len(sid), CHUNK_TOK):
                windows.append(sid[s:s+CHUNK_TOK])
                if len(windows) >= MAX_CHUNKS: break
            i += 1
            if len(windows) >= MAX_CHUNKS: break
            continue
        if cur and cur_len + len(sid) > CHUNK_TOK:
            windows.append(cur)
            if len(windows) >= MAX_CHUNKS: break
            carry = cur_sents[-OVERLAP_SENTS:] if OVERLAP_SENTS else []
            cur = [t for cs in carry for t in cs]
            cur_len = len(cur); cur_sents = list(carry)
        else:
            if not cur: cur_sents = []
        cur += sid; cur_len += len(sid); cur_sents.append(sid); i += 1
    if cur and len(windows) < MAX_CHUNKS:
        windows.append(cur)

    cls, sep = tokenizer.cls_token_id, tokenizer.sep_token_id
    return [[cls] + w[:CHUNK_TOK] + [sep] for w in windows[:MAX_CHUNKS]]

def encode_document(model, windows, tokenizer, device):
    maxlen = max(len(w) for w in windows)
    pad = tokenizer.pad_token_id
    ids = torch.full((len(windows), maxlen), pad, dtype=torch.long)
    mask = torch.zeros((len(windows), maxlen), dtype=torch.long)
    for j, w in enumerate(windows):
        ids[j, :len(w)] = torch.tensor(w); mask[j, :len(w)] = 1
    ids, mask = ids.to(device), mask.to(device)
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0, :]  # [n, 768]
    return out.mean(dim=0)                                                            # [768]

print('shared chunking + document encoder ready')

shared chunking + document encoder ready


## Per-model preprocessing

Two small differences from NB5b, both because these encoders were pre-trained without Farasa:

- **AraELECTRA:** `ArabertPreprocessor(model_name='aubmindlab/araelectra-base')` does the same
  punctuation and diacritic normalization the model was pre-trained with, without the segmenter
  step. Same one-shot preprocess-and-cache pattern as NB5b.
- **CAMeLBERT-MSA:** the model does not need any external preprocessor; I apply a light clean that
  normalizes tatweel and repeated whitespace, matching what CAMeL Tools does upstream in most
  pipelines.

In [3]:
from arabert.preprocess import ArabertPreprocessor

def prep_araelectra(text):
    # cached separately so we can reuse it if we come back to AraELECTRA later
    return _ae_prep.preprocess(text)

def prep_camelbert(text):
    # light MSA clean: strip tatweel, unify whitespace, keep Arabic + punctuation as-is
    t = re.sub('\u0640', '', str(text))          # tatweel
    t = re.sub(r'\s+', ' ', t).strip()
    return t

_ae_prep = ArabertPreprocessor(model_name='aubmindlab/araelectra-base')
print('per-model preprocessors ready')

per-model preprocessors ready


/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:274: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN = re.compile(u"([^\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:276: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN_SPLIT = re.compile(u"([\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:281: SyntaxWarning: invalid escape sequence '\s'
  ARABIC_STRING = re.compile(u"([^\u0600-\u0652%s%s%s\s\d])" \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1237: SyntaxWarning: invalid escape sequence '\s'
  u"(?<=\s(%s|%s))%s" % (WAW, YEH, FATHA), \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1450: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(u"(?<=[\s\d])([%s])+"%(TASHKEEL_STRING),"",text,  re.UNICODE)


## Extract [CLS] for a single model — reusable routine

Takes an encoder id and its preprocessor, loads the tokenizer and model, freezes it, and produces
one R^768 vector per article. Caches the vectors as an `.npy` per model so I can re-run the probe
without re-encoding. If the cache exists, it is loaded directly.

In [4]:
from transformers import AutoTokenizer, AutoModel

def build_frozen_embeddings(encoder_id, preprocess_fn, cache_path):
    if os.path.exists(cache_path):
        emb = np.load(cache_path)
        print(f'  loaded cached {cache_path} shape={emb.shape}')
        return emb

    tok = AutoTokenizer.from_pretrained(encoder_id)
    mdl = AutoModel.from_pretrained(encoder_id).to(DEVICE).eval()
    for p in mdl.parameters(): p.requires_grad_(False)

    t0 = time.time()
    prep = df['text'].apply(preprocess_fn)
    print(f'  preprocessed {len(prep)} articles in {time.time()-t0:.0f}s')

    emb = np.zeros((len(df), 768), dtype=np.float32)
    t0 = time.time()
    for i in range(len(df)):
        windows = chunk_token_windows(prep.iloc[i], tok)
        emb[i] = encode_document(mdl, windows, tok, DEVICE).cpu().numpy()
        if (i+1) % 500 == 0 or i+1 == len(df):
            el = time.time()-t0
            print(f'  [{i+1:5d}/{len(df)}] {el/(i+1):.2f}s/doc '
                  f'| ETA {el/(i+1)*(len(df)-i-1)/60:.0f}m', flush=True)
    np.save(cache_path, emb)
    del mdl; torch.cuda.empty_cache() if DEVICE=='cuda' else None
    print(f'  saved {cache_path}')
    return emb

print('encoder routine ready')

encoder routine ready


## Linear probe scoring — reusable

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

def split_mask(name): return (df['split'] == name).to_numpy()
ytr = df.loc[split_mask('train'), 'label'].to_numpy()
yte = df.loc[split_mask('test'),  'label'].to_numpy()

def probe_on(embeddings, name):
    Etr = embeddings[split_mask('train')]
    Ete = embeddings[split_mask('test')]
    clf = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
    clf.fit(Etr, ytr)
    pred  = clf.predict(Ete)
    proba = clf.predict_proba(Ete)[:, 1]
    row = {
        'model':      name,
        'accuracy':   accuracy_score(yte, pred),
        'precision':  precision_score(yte, pred),
        'recall':     recall_score(yte, pred),
        'macro_f1':   f1_score(yte, pred, average='macro'),
        'auc_roc':    roc_auc_score(yte, proba),
    }
    return row, pred, proba

print('probe routine ready')

probe routine ready


## Run — AraELECTRA

In [6]:
ae_cache = f'{OUT_DIR}/nb5c_araelectra_cls_frozen.npy'
ae_emb = build_frozen_embeddings('aubmindlab/araelectra-base-discriminator',
                                 prep_araelectra, ae_cache)
ae_row, ae_pred, ae_proba = probe_on(ae_emb, 'AraELECTRA frozen + probe')
print('AraELECTRA frozen + probe:')
for k in ['accuracy','precision','recall','macro_f1','auc_roc']:
    print(f'  {k:<10} {100*ae_row[k]:.1f}')

config.json:   0%|          | 0.00/503 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: aubmindlab/araelectra-base-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  preprocessed 7101 articles in 26s


Token indices sequence length is longer than the specified maximum sequence length for this model (1041 > 512). Running this sequence through the model will result in indexing errors


  [  500/7101] 0.07s/doc | ETA 8m
  [ 1000/7101] 0.07s/doc | ETA 7m
  [ 1500/7101] 0.07s/doc | ETA 7m
  [ 2000/7101] 0.07s/doc | ETA 6m
  [ 2500/7101] 0.07s/doc | ETA 6m
  [ 3000/7101] 0.07s/doc | ETA 5m
  [ 3500/7101] 0.07s/doc | ETA 4m
  [ 4000/7101] 0.07s/doc | ETA 4m
  [ 4500/7101] 0.07s/doc | ETA 3m
  [ 5000/7101] 0.07s/doc | ETA 3m
  [ 5500/7101] 0.07s/doc | ETA 2m
  [ 6000/7101] 0.07s/doc | ETA 1m
  [ 6500/7101] 0.07s/doc | ETA 1m
  [ 7000/7101] 0.08s/doc | ETA 0m
  [ 7101/7101] 0.08s/doc | ETA 0m
  saved /kaggle/working/nb5c_araelectra_cls_frozen.npy
AraELECTRA frozen + probe:
  accuracy   99.5
  precision  99.6
  recall     99.5
  macro_f1   99.5
  auc_roc    99.9


## Run — CAMeLBERT-MSA

In [7]:
cb_cache = f'{OUT_DIR}/nb5c_camelbert_msa_cls_frozen.npy'
cb_emb = build_frozen_embeddings('CAMeL-Lab/bert-base-arabic-camelbert-msa',
                                 prep_camelbert, cb_cache)
cb_row, cb_pred, cb_proba = probe_on(cb_emb, 'CAMeLBERT-MSA frozen + probe')
print('CAMeLBERT-MSA frozen + probe:')
for k in ['accuracy','precision','recall','macro_f1','auc_roc']:
    print(f'  {k:<10} {100*cb_row[k]:.1f}')

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

  preprocessed 7101 articles in 3s
  [  500/7101] 0.08s/doc | ETA 9m
  [ 1000/7101] 0.08s/doc | ETA 8m
  [ 1500/7101] 0.08s/doc | ETA 8m
  [ 2000/7101] 0.08s/doc | ETA 7m
  [ 2500/7101] 0.08s/doc | ETA 6m
  [ 3000/7101] 0.08s/doc | ETA 5m
  [ 3500/7101] 0.08s/doc | ETA 5m
  [ 4000/7101] 0.08s/doc | ETA 4m
  [ 4500/7101] 0.08s/doc | ETA 3m
  [ 5000/7101] 0.08s/doc | ETA 3m
  [ 5500/7101] 0.08s/doc | ETA 2m
  [ 6000/7101] 0.08s/doc | ETA 1m
  [ 6500/7101] 0.08s/doc | ETA 1m
  [ 7000/7101] 0.08s/doc | ETA 0m
  [ 7101/7101] 0.08s/doc | ETA 0m
  saved /kaggle/working/nb5c_camelbert_msa_cls_frozen.npy
CAMeLBERT-MSA frozen + probe:
  accuracy   99.7
  precision  99.8
  recall     99.6
  macro_f1   99.7
  auc_roc    100.0


## Combined results table — three encoders, same protocol

In [8]:
from_nb5b = {
    'model':      'AraBERTv2 frozen + probe',
    'accuracy':   0.995, 'precision': 0.995, 'recall': 0.995,
    'macro_f1':   0.995, 'auc_roc':   1.000,
}
res = pd.DataFrame([from_nb5b, ae_row, cb_row])
show = res.copy()
for c in ['accuracy','precision','recall','macro_f1','auc_roc']:
    show[c] = (100 * show[c]).round(1)
print(show.to_string(index=False))
print()
best = res.loc[res['macro_f1'].idxmax()]
print(f"best frozen probe: {best['model']} — macro-F1 {100*best['macro_f1']:.1f}%")
print()
print('reference points:')
print("  NB5a statistical-only:            84.7%")
print("  NB3 cheap-signal control:         68.8%")

res.to_parquet(f'{OUT_DIR}/nb5c_results.parquet', index=False)
print('\nsaved nb5c_results.parquet')

                       model  accuracy  precision  recall  macro_f1  auc_roc
    AraBERTv2 frozen + probe      99.5       99.5    99.5      99.5    100.0
   AraELECTRA frozen + probe      99.5       99.6    99.5      99.5     99.9
CAMeLBERT-MSA frozen + probe      99.7       99.8    99.6      99.7    100.0

best frozen probe: CAMeLBERT-MSA frozen + probe — macro-F1 99.7%

reference points:
  NB5a statistical-only:            84.7%
  NB3 cheap-signal control:         68.8%

saved nb5c_results.parquet


## Per-generator recall — does the ranking change across encoders?

Same table as NB5b, computed for both new encoders on their test-split predictions. This exposes
whether AraELECTRA and CAMeLBERT find the same generators easy or hard as AraBERT did — which is
what the encoder-choice discussion will hinge on.

In [9]:
te = df[split_mask('test')].reset_index(drop=True)
gen_table = {'AraELECTRA': ae_pred, 'CAMeLBERT-MSA': cb_pred}
gens = sorted(te.loc[te['label']==1, 'generator'].unique().tolist())

print(f"{'generator':<12}" + ''.join(f"{name:>16}" for name in gen_table))
for g in gens:
    row = f'{g:<12}'
    for name, pred in gen_table.items():
        sub = te[(te['label']==1) & (te['generator']==g)]
        rec = (pred[sub.index] == 1).mean()
        row += f'{100*rec:>15.0f}%'
    print(row)

# human class recall for each model
row = f'{"HUMAN":<12}'
for name, pred in gen_table.items():
    sub = te[te['label']==0]
    rec = (pred[sub.index] == 0).mean()
    row += f'{100*rec:>15.0f}%'
print(row)

generator         AraELECTRA   CAMeLBERT-MSA
deepseek                 99%             99%
gemini                  100%            100%
gpt                     100%            100%
opus                    100%            100%
qwen                     99%            100%
sonnet                   99%            100%
HUMAN                   100%            100%


## LOGO on each encoder (frozen [CLS], six folds each)

Same routine as the AraBERT LOGO check, applied to the two new encoders. This is the honest test of
generalization: if a frozen encoder’s number came from seeing every generator in training, holding
one out will drop it. If the encoder learned a genuine human/AI distinction, the drop is small.

In [10]:
def rows_train_holdout(g):
    m = (df['split'] == 'train') & ((df['label']==0) | (df['generator'] != g))
    return np.where(m)[0]
def rows_test_holdout(g):
    m = (df['split'] == 'test') & ((df['label']==0) | (df['generator'] == g))
    return np.where(m)[0]

def logo(emb, name):
    out = []
    for g in gens:
        tr = rows_train_holdout(g); te_i = rows_test_holdout(g)
        Xtr, y_tr = emb[tr], df['label'].iloc[tr].to_numpy()
        Xte, y_te = emb[te_i], df['label'].iloc[te_i].to_numpy()
        clf = LogisticRegression(max_iter=5000, class_weight='balanced',
                                 random_state=SEED).fit(Xtr, y_tr)
        pred = clf.predict(Xte); proba = clf.predict_proba(Xte)[:, 1]
        cm = confusion_matrix(y_te, pred, labels=[0, 1])
        out.append({'model': name, 'held_out': g,
                    'n_test_ai':    int((y_te==1).sum()),
                    'ai_recall':    cm[1,1]/max(cm[1].sum(),1),
                    'human_recall': cm[0,0]/max(cm[0].sum(),1),
                    'macro_f1':     f1_score(y_te, pred, average='macro'),
                    'auc_roc':      roc_auc_score(y_te, proba)})
    return out

logo_ae = logo(ae_emb, 'AraELECTRA')
logo_cb = logo(cb_emb, 'CAMeLBERT-MSA')
logo_df = pd.DataFrame(logo_ae + logo_cb)

# summary per model
for name in ['AraELECTRA', 'CAMeLBERT-MSA']:
    sub = logo_df[logo_df['model'] == name]
    print(f"\n{name} LOGO:")
    print(f"  mean macro-F1  {100*sub['macro_f1'].mean():.1f}%")
    print(f"  std            {100*sub['macro_f1'].std():.1f}%")
    print(f"  worst          {100*sub['macro_f1'].min():.1f}%  "
          f"({sub.loc[sub['macro_f1'].idxmin(),'held_out']} held out)")

print('\nAraBERTv2 LOGO from NB5b-LOGO for reference:')
print('  mean 97.6% | std 2.4% | worst 92.8% (gpt held out)')

logo_df.to_parquet(f'{OUT_DIR}/nb5c_logo.parquet', index=False)


AraELECTRA LOGO:
  mean macro-F1  96.2%
  std            6.2%
  worst          83.5%  (gpt held out)

CAMeLBERT-MSA LOGO:
  mean macro-F1  98.1%
  std            2.8%
  worst          92.7%  (gpt held out)

AraBERTv2 LOGO from NB5b-LOGO for reference:
  mean 97.6% | std 2.4% | worst 92.8% (gpt held out)


## Notes

**Why this notebook keeps everything except the encoder constant.** The point is a controlled
comparison. Preprocessing, chunking, pooling, the classifier head, the split, and even the random
seed are byte-identical to NB5b. Any performance difference between the three probes therefore comes
from the encoder — which is exactly the question the thesis needs answered before choosing a hybrid
core.

**AraELECTRA vs AraBERTv2.** Both were pre-trained by AUB MIND on the same 77 GB corpus, so this
comparison isolates the effect of the pre-training objective (discriminative vs masked-LM). A tie or
close race is the expected outcome; a large gap either way is the finding worth reporting.

**CAMeLBERT-MSA.** Trained on MSA news specifically, which is exactly this corpus's sub-genre. It is
the strongest prior candidate for the news-detection task on that basis, and probing it here quantifies
whether that domain match translates into a better linear-separability of human vs AI in feature space.

**Cost.** Both encoders are BERT-base size and neither uses Farasa, so this notebook should run
substantially faster than NB5b — encoding time roughly 10 minutes per model on a T4 (versus about
23 minutes for AraBERTv2 including Farasa).

**Reading LOGO alongside the seen-all probe.** A frozen probe scoring 99% on the standard split but
falling to 90% under LOGO would tell a different story than one that stays at 97%. The pair
(seen-all, LOGO mean, LOGO worst) is the honest report; the thesis should present all three side by
side for the three encoders and let the reader choose which robustness profile matters.